# OUSIA Phase 0 — Incremental Expansion (A100)
Adds OpenHermes-2.5 + OASST1 on top of existing OUSIA-Phase1 adapter
Base: Qwen/Qwen3.5-4B | LoRA: r=16, seq=1024, batch=4, grad_accum=4
Runtime: A100 40GB GPU | Est. ~45 min

## 1. Setup

In [ ]:
# Upgrade transformers
!pip install -q --upgrade transformers huggingface_hub
!pip install -q transformers peft datasets accelerate bitsandbytes torch trl
!pip install -q tensorboard google-cloud-storage

In [ ]:
# Clone repo
!git clone https://github.com/plntrprotocol/aureth-training.git /content/aureth-training 2>/dev/null || echo "already mounted"
%cd /content/aureth-training

In [ ]:
# HuggingFace token
from google.colab import userdata
import os
hf_token = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = hf_token
print(f"HF_TOKEN set ({len(hf_token)} chars)")

## 2. Upload OUSIA-Phase1 Adapter
1. Download `ousia-phase1-adapter.zip` from your Phase 1 Colab session
2. Upload it to this Colab (Files panel → Upload)

In [ ]:
import os
adapter_zip = "/content/ousia-phase1-adapter.zip"
if os.path.exists(adapter_zip):
    print(f"Found: {adapter_zip}")
    !unzip -o {adapter_zip} -d /content/ousia-phase1-adapter/
    print("Unzipped OK")
else:
    raise FileNotFoundError("ousia-phase1-adapter.zip not found — upload it first")

## 3. Load Base Model + Phase1 Adapter

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch
BASE_MODEL_ID = "Qwen/Qwen3.5-4B"
ADAPTER_PATH = "/content/ousia-phase1-adapter"
print(f"Loading tokenizer: {BASE_MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, token=hf_token)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"Tokenizer: {tokenizer.vocab_size} tokens")
print(f"Loading base model (4bit QLoRA)...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    token=hf_token,
    device_map="cuda",
    load_in_4bit=True,
    torch_dtype=torch.bfloat16,
)
print(f"Base model: {base_model.num_parameters():,} params")
print(f"Loading Phase1 adapter...")
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.print_trainable_parameters()

## 4. Load Datasets

In [ ]:
from datasets import load_dataset, concatenate_datasets
MAX_SEQ = 1024
print("Loading OpenHermes-2.5...")
openhermes = load_dataset("mistralai/OpenHermes-2.5", split="train", streaming=True)
print("Loading OASST1 English...")
oasst = load_dataset("OpenAssistant/oasst1", subset="lang_en", split="train", streaming=True)
print("Datasets loaded")

In [ ]:
def tokenize(example):
    text = example.get("text") or example.get("conversation") or str(example)
    if not text or len(text.strip()) < 10:
        return {"input_ids": [], "labels": []}
    tokens = tokenizer(text, truncation=True, max_length=MAX_SEQ, padding="max_length")
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens
oh_sample = openhermes.take(20000)
oh_tokenized = oh_sample.map(tokenize, remove_columns=oh_sample.column_names)
print(f"OpenHermes: {len(oh_tokenized)} examples")
oasst_sample = oasst.take(15000)
oasst_tokenized = oasst_sample.map(tokenize, remove_columns=oasst_sample.column_names)
print(f"OASST1: {len(oasst_tokenized)} examples")
combined = concatenate_datasets([oh_tokenized, oasst_tokenized])
print(f"Combined: {len(combined)} examples")

## 5. Training

In [ ]:
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from trl import SFTTrainer
import os
OUTPUT_DIR = "/content/output/ousia-phase0"
os.makedirs(OUTPUT_DIR, exist_ok=True)
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,     # effective batch = 16
    num_train_epochs=2,
    learning_rate=1e-4,             # lower LR preserves existing adapter
    warmup_ratio=0.03,
    weight_decay=0.01,
    max_grad_norm=0.5,
    logging_steps=50,
    save_steps=200,
    save_total_limit=2,
    bf16=True,                      # A100: BF16
    optim="paged_adamw_8bit",
    dataloader_num_workers=2,
    report_to=["tensorboard"],
    remove_unused_columns=False,
)
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=combined,
    data_collator=data_collator,
    tokenizer=tokenizer,
    max_seq_length=MAX_SEQ,
)
print(f"A100 training: batch=4, seq={MAX_SEQ}, eff_batch=16")
print(f"{len(combined)} examples, ~45 min")
trainer.train()

## 6. Save Adapter

In [ ]:
adapter_path = "/content/ousia-phase0-adapter"
trainer.save_model(adapter_path)
print(f"Saved: {adapter_path}")
!cd /content && zip -r ousia-phase0-adapter.zip ousia-phase0-adapter/
print("Zip: ousia-phase0-adapter.zip — download from Files panel")